In [ ]:
import math
import numpy as np
import pandas as pd
from datetime import datetime
import time as time_mod

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from pyomo.environ import *
from pyomo.dae import *

from scipy.integrate import odeint, solve_ivp
from scipy.interpolate import CubicSpline
from sklearn.metrics import r2_score, mean_squared_error

from joblib import Parallel, delayed

import pickle
import random

import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import gc

# pytorch relates imports
import torch
import torch.nn as nn
import torch.optim as optim

# imports from captum library
from captum.attr import IntegratedGradients, Saliency, Occlusion

# Generate Synthetic Raman Data

In [ ]:
def case_study1(t, y, Ki):
    Ca, Cb, Cc = y
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    v1max = 4.5
    v2max = 2.5
    
    Km1 = 5
    Km2 = 5
    
    Ki1 = Ki[0]
    Ki2 = Ki[1]
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    inh1 = (1 + (Cc/Ki1))
    inh2 = (1 + (Cc/Ki2))
    
#     # Uncompetitive Inhibition    
#     v1 = ((v1max/inh1)*Ca)/((Ca + Km1*inh1))
#     v2 = ((v2max/inh2)*Cb)/((Ca + Km2*inh2))
    
    # Noncompetitive Inhibition    
    v1 = (v1max*Ca)/((Ca + Km1)*inh1)
    v2 = (v2max*Cb)/((Cb + Km2)*inh2)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    dCadt = -v1
    dCbdt = v1 - v2
    dCcdt = v2
    
    return [dCadt, dCbdt, dCcdt]

# Coupled Raman Hybrid Methodology

In [ ]:
def layer_calculation(model, vector, layer_int, num_output_nodes, no_actf = 0):
    
    output = []
    
    for i in range(num_output_nodes):
        temp = model.b[layer_int, i]
        
        for k in range(len(vector)):
            temp += model.W[layer_int, k, i]*vector[k]
        
        if no_actf == 0:
            #h_act = (math.e**(2*temp) - 1) / (math.e**(2*temp) + 1)
            h_act = 1 / (1 + exp(-1 * temp))
        else:
            h_act = temp
        h = h_act
        output.append(h)

    return output
    

def ANN_inhibition_model(model, d, t):

    # Extracting input variables from the Pyomo model
    Ca = model.conc_Ca[d, t]
    Cb = model.conc_Cb[d, t]
    Cc = model.conc_Cc[d, t]
    
    input_vector = [Ca, Cb, Cc]
    
    #print(len(model.ANN_arch))
    for i in range(len(model.ANN_arch)-2):
        output_vector = layer_calculation(model, input_vector, i, model.ANN_arch[i+1])
        input_vector = output_vector

    j = i + 1
    output_vector = layer_calculation(model, input_vector, j, model.ANN_arch[j+1])
    
    fin = output_vector
    
    return fin

In [ ]:
def dCadt_int_con(model, d, t):
    if t == 0:
        return model.conc_Ca[d, t] == model.conc_Ca_initial[d]
    
    Ezk1 = ANN_inhibition_model(model, d, t)[0]
    v1 = Ezk1*model.v1m*model.conc_Ca[d,t]
    
    return model.dCadt[d, t] == -1*v1


def dCbdt_int_con(model, d, t):
    if t == 0:
        return model.conc_Cb[d, t] == model.conc_Cb_initial[d]
    
    Ezk1 = ANN_inhibition_model(model, d, t)[0]
    v1 = Ezk1*model.v1m*model.conc_Ca[d,t]
    
    Ezk2 = ANN_inhibition_model(model, d, t)[1]
    v2 = Ezk2*model.v2m*model.conc_Cb[d,t]
    
    return model.dCbdt[d, t] == v1 - v2


def dCcdt_int_con(model, d, t):
    if t == 0:
        return model.conc_Cc[d, t] == model.conc_Cc_initial[d]
    
    Ezk2 = ANN_inhibition_model(model, d, t)[1]
    v2 = Ezk2*model.v2m*model.conc_Cb[d,t]
    
    return model.dCcdt[d, t] == v2

In [ ]:
def sp_c_Ca(model, d, t, wv):
    return model.sp_con_Ca[d, t, wv] == model.pure_Sa[wv]*model.conc_Ca[d, t]

def sp_c_Cc(model, d, t, wv):
    return model.sp_con_Cc[d, t, wv] == model.pure_Sc[wv]*model.conc_Cc[d, t]

In [ ]:
def D_mse_objective(model):
    fin = 0
    
    for i in model.d_idx:
        for j in model.t_meas:
            for k in model.wv_meas:
                fin += (model.D_final_meas[i,j,k] - model.sp_con_Ca[i,j,k] - model.sp_con_Cc[i,j,k])**2
                
    final = fin/(len(model.d_idx)*len(model.t_meas)*len(model.wv_meas))
    return final

In [ ]:
def coupled_Raman_formulation(t_fin, wv_fin, D_full, C0, ANN_arch):
    
    nd = len(C0)
    nt = len(t_fin)
    nw = len(wv_fin)

    D_dict = {}

    for i in range(nd):
        for j in range(nt):
            for k in range(nw):
                l = i*(nt) + j
                D_dict[(i, t_fin[j], wv_fin[k])] = D_full[l, k]

    Ca_initial = {}
    Cb_initial = {}
    Cc_initial = {}

    for i in range(nd):
        Ca_initial[i] = C0[i,0]
        Cb_initial[i] = C0[i,1]
        Cc_initial[i] = C0[i,2]
    
    ##~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    ##~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    model = ConcreteModel()

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Data Related Sets - experiment number, time point and wavenumber

    model.d_idx = Set(initialize = range(nd))
    model.t_meas = Set(initialize = t_fin)
    model.wv_meas = Set(initialize = wv_fin)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # ANN Related Sets - 1 layer - Number of nodes and number of inputs


    W_index_set = []
    b_index_set = []

    for i in range(len(ANN_arch)-1):
        j = ANN_arch[i]
        k = ANN_arch[i+1]

        for r in range(j):
            for c in range(k):
                W_index_set.append(tuple([i, r, c]))

    for i in range(len(ANN_arch)-1):
        j = ANN_arch[i+1]

        for r in range(j):
            b_index_set.append(tuple([i, r]))


    model.W_index = Set(initialize=W_index_set, dimen=3)
    model.b_index = Set(initialize=b_index_set, dimen=2)

    model.ANN_arch = Param(range(len(ANN_arch)), initialize=ANN_arch)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Parameters for this formulation - Measured Concentration and Known Etot conc

    model.D_final_meas = Param(model.d_idx, model.t_meas, model.wv_meas, initialize = D_dict)

    model.conc_Ca_initial = Param(model.d_idx, initialize = Ca_initial)
    model.conc_Cb_initial = Param(model.d_idx, initialize = Cb_initial)
    model.conc_Cc_initial = Param(model.d_idx, initialize = Cc_initial)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Variables for species and their derivative variables for each respectibe species

    model.pure_Sa = Var(model.wv_meas, within = NonNegativeReals, bounds = (0, 100))
    model.pure_Sc = Var(model.wv_meas, within = NonNegativeReals, bounds = (0, 100))

    model.sp_con_Ca = Var(model.d_idx, model.t_meas, model.wv_meas, within = NonNegativeReals)
    model.sp_con_Cc = Var(model.d_idx, model.t_meas, model.wv_meas, within = NonNegativeReals)

    model.time = ContinuousSet(initialize=model.t_meas, bounds=(t_fin[0], t_fin[-1]))

    model.conc_Ca = Var(model.d_idx, model.time, within = NonNegativeReals)
    model.conc_Cb = Var(model.d_idx, model.time, within = NonNegativeReals)
    model.conc_Cc = Var(model.d_idx, model.time, within = NonNegativeReals)

    model.dCadt = DerivativeVar(model.conc_Ca, within = Reals)
    model.dCbdt = DerivativeVar(model.conc_Cb, within = Reals)
    model.dCcdt = DerivativeVar(model.conc_Cc, within = Reals)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Variables for kinetic Parameters within the ODE kinetic system

    model.v1m = Var(within = NonNegativeReals)
    model.v2m = Var(within = NonNegativeReals)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Variables for ANN weights and biases (used in growth rate calculations)

    model.W = Var(model.W_index, bounds=(-1, 1))
    model.b = Var(model.b_index, bounds=(-1, 1))

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Spectral Constraints

    model.spectral_Ca = Constraint(model.d_idx, model.t_meas, model.wv_meas, rule = sp_c_Ca)
    model.spectral_Cc = Constraint(model.d_idx, model.t_meas, model.wv_meas, rule = sp_c_Cc)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Mechanistic Constraints

    model.deriv_Ca = Constraint(model.d_idx, model.time, rule = dCadt_int_con)
    model.deriv_Cb = Constraint(model.d_idx, model.time, rule = dCbdt_int_con)
    model.deriv_Cc = Constraint(model.d_idx, model.time, rule = dCcdt_int_con)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Objective function

    model.obj = Objective(expr = D_mse_objective)

    # # Discretizer for Pyomo.DAE (Differential-Algebraic Equations)
    discretizer = TransformationFactory('dae.collocation')
    discretizer.apply_to(model, nfe=len(t_fin), ncp=2, scheme='LAGRANGE-RADAU')  # Apply Lagrange-Radau collocation method

    solver = SolverFactory("multistart")

    results = solver.solve(
        model,
        suppress_unbounded_warning=True,
        iterations=50,
        solver="ipopt",
        solver_args={"options": {"max_cpu_time": 30.0, 
                                 "max_iter": 3000}})

    # Print solver status and results
    print("Solver status:", results.solver.status)
    print("Termination Condition:", results.solver.termination_condition)
    print("Objective: ", value(model.obj))  # Print the objective function value
    
    return model

# Hybrid Model and Recovered Spectra Testing

In [ ]:
def ANN_TIV_param(model, ANN_arch):

    ANN_params = {}

    for i in range(len(ANN_arch)-1):
        key = "W" + str(i+1)

        r = ANN_arch[i]
        c = ANN_arch[i+1]

        np_temp = np.zeros((r, c))

        for j in range(r):
            for k in range(c):
                np_temp[j,k] = value(model.W[i, j, k])

        ANN_params[key] = np_temp

    ##~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    ##~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    for i in range(len(ANN_arch)-1):
        key = "b" + str(i+1)

        r = ANN_arch[i+1]
        np_temp = np.zeros((1,r))

        for j in range(r):
            np_temp[0,j] = value(model.b[i,j])

        ANN_params[key] = np_temp
    
    ##~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    ##~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    K_params = {}

    K_params["k1"] = value(model.v1m)
    K_params["k2"] = value(model.v2m)
    
    return [ANN_params, K_params]

In [ ]:
def ANN_Ezkin_model(ANN_arch, ANN_params):
    arch_list = []

    for i in range(len(ANN_arch)-1):
        inp = ANN_arch[i]
        outp = ANN_arch[i+1]

        arch_list.append(nn.Linear(inp, outp))
        arch_list.append(nn.Sigmoid())

    ANN_model = nn.Sequential(*arch_list)

    ANN_recon_dict = {}

    temp_dict = {}

    for i in range(len(ANN_arch)-1):
        j = i + 1
        k = 2*i

        Wname = "W" + str(j)
        bname = "b" + str(j)

        with torch.no_grad():
            ANN_model[k].weight.copy_(torch.from_numpy(ANN_params[Wname].T))
            ANN_model[k].bias.copy_(torch.from_numpy(ANN_params[bname].reshape(ANN_arch[j],)))
            
    
    return ANN_model

In [ ]:
def HMI_CS1_system(t, y, ANN_model, K_params):
    
    c1, c2, c3 = y
    
    x_tensor = torch.tensor(np.array([c1, c2, c3]), dtype=torch.float32)
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    Ezkin1 = ANN_model(x_tensor).detach().numpy()[0]
    Ezkin2 = ANN_model(x_tensor).detach().numpy()[1]
    
    k1 = K_params["k1"]
    k2 = K_params["k2"]
    
    r1 = k1*Ezkin1*c1
    r2 = k2*Ezkin2*c2
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~    

    dc1dt = -1*r1
    
    dc2dt = r1 - r2
    
    dc3dt = r2
        
    return [dc1dt, dc2dt, dc3dt]

In [ ]:
def predictive_perf_testing(ic, tspan, Ki, ANN_arch, ANN_params, K_params):
    
    ann = ANN_Ezkin_model(ANN_arch, ANN_params)
    
    r2_list = []
    rmse_list = []
    nrmse_list = []

    sol_true = solve_ivp(case_study1, 
                    tspan, 
                    ic,
                    args = Ki,
                    method = "Radau")
    
    teval = sol_true.t

    sol_calc = solve_ivp(HMI_CS1_system, 
                        tspan, 
                        ic, 
                        t_eval = teval, 
                        args = tuple([ann, K_params]), 
                        method = "Radau")

    y_true = sol_true.y
    y_calc = sol_calc.y

    for j in range(len(y_true)):
        mse = mean_squared_error(y_true[j], y_calc[j])
        rmse_list.append(mse**0.5)
        nrmse_list.append((mse**0.5)/np.mean(y_true[j]))
        
        if r2_score(y_true[j], y_calc[j]) < 0:
            r2_list.append(0)
        else:
            r2_list.append(r2_score(y_true[j], y_calc[j]))
        
    return tuple([np.mean(r2_list), np.mean(rmse_list), np.mean(nrmse_list)])

In [ ]:
def spectra_recovery_metrics(ST_ground, model):
    
    groundtruth_spectra = ST_ground
    
    ps_list = [model.pure_Sa, model.pure_Sc]
    recovered_spectra = []
    for i in range(len(ps_list)):
        spc = []
        for j in model.wv_meas:
            spc.append(value(ps_list[i][value(j)]))
            
        spc = np.array(spc)
        recovered_spectra.append(spc)
    recovered_spectra = np.array(recovered_spectra)
    
    spectral_metric = {}
    
    for i in range(len(groundtruth_spectra)):    
        gs = groundtruth_spectra[i,:]
        gs_n = gs / np.linalg.norm(gs)

        rs = recovered_spectra[i,:] 
        rs_n = rs / np.linalg.norm(rs)

        err = gs - rs

        sm = np.matmul(gs_n, rs_n.T)
        nrmse = np.linalg.norm(err)/np.linalg.norm(gs)
        
        ky = "Component " + str(i+1)
        spectral_metric[ky] = [sm, nrmse]
    
    return spectral_metric

In [ ]:
C1_range = np.logspace(-2, 1, 50)

rect_init = []

for i in range(len(C1_range)):
    rect_init.append(np.array([C1_range[i], 0, 0]))

In [ ]:
train_scenario3 = os.listdir("Train Conditions/Scenario 3")

In [ ]:
for i in range(len(train_scenario3)):

    file = os.path.join("Train Conditions/Scenario 3", train_scenario3[i])

    with open(file, "rb") as f:
        full_data = pickle.load(f)

    t_fin = full_data["t_fin"]
    wv_fin = full_data["wv_fin"].reshape(-1,)
    D_full = full_data["D_full"]
    C0 = full_data["C0"]
    ST_ground = full_data["ST_ground"]

    ANN_arch = [3, 2, 2]

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    print("\033[1mTime start of Opt: \033[0m", datetime.now())

    start_compute = time_mod.time()

    model = coupled_Raman_formulation(t_fin, wv_fin, D_full, C0, ANN_arch)

    end_compute = time_mod.time()

    print("\033[1mTime end of Opt: \033[0m", datetime.now())

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    compute_time = end_compute - start_compute

    ANN_params, K_params = ANN_TIV_param(model, ANN_arch)

    rs_ST = int(train_scenario3[i].split("_")[3])
    rs_D = float(train_scenario3[i].split("_")[-1].split(".")[0])

    Ki1 = float(train_scenario3[i].split("_")[1])
    Ki2 = float(train_scenario3[i].split("_")[2])
    Ki = tuple([[Ki1, Ki2]])
    spc_ovp = float(train_scenario3[i].split("_")[4])

    test_input = []
    for m in range(len(rect_init)):
        test_input.append(tuple([rect_init[m], (0, 20), Ki, ANN_arch, ANN_params, K_params]))

    conc_predictive_performance = Parallel(n_jobs=-1)(
        delayed(predictive_perf_testing)(ic, t, k, AA, AP, KP) for ic, t, k, AA, AP, KP in test_input)

    spectra_predictive_performance = spectra_recovery_metrics(ST_ground, model)

    final_performance_dict = {}
    final_performance_dict["conc_perf"] = conc_predictive_performance
    final_performance_dict["spectra_perf"] = spectra_predictive_performance
    final_performance_dict["ANN model"] = [ANN_arch, ANN_params]
    final_performance_dict["K params"] = [K_params]
    final_performance_dict["Compute Time"] = compute_time

    print("\033[1mTime end of Test: \033[0m", datetime.now())

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    path_name = "Coupled Performance/Hybrid model/Scenario 3/(322)_ANN/"
    file_name = "Perf_"+str(Ki[0][0])+"_"+str(Ki[0][1])+"_"+str(rs_ST)+"_"+str(spc_ovp)+"_"+str(rs_D)+".pkl"
    final_name = path_name + file_name

    with open(final_name, "wb") as f:
        pickle.dump(final_performance_dict, f)

    del model
    gc.collect()

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")